## COMP90024 Group 25 Visualization

Here are the visualisation of our five scenarios, you can click on the button to see the plots.

In [1]:
# Store the data and process the data in the jupyter notebook for pedcount scenario since it is too large
import pandas as pd
import requests
from sklearn.preprocessing import MinMaxScaler
from scipy import stats
file_path = "pedestrian-counting-system-monthly-counts-per-hour.json"
try:
    df = pd.read_json(file_path)    
    df[['lon', 'lat']] = df['location'].apply(pd.Series)
    
    df.drop(columns=['location'], inplace=True)
    
except ValueError as e:
    print("Error reading the JSON file:", e)
except FileNotFoundError as e:
    print("File not found:", e)
df_pedcountperh = df.drop(columns=[ 'locationid', 'direction_1', 'direction_2'])
df = df_pedcountperh
z_scores = stats.zscore(df['total_of_directions'])
abs_z_scores = abs(z_scores)
filtered_entries = (abs_z_scores < 3)  # 通常 z-score 大于 3 的数据点被认为是离群值
df = df[filtered_entries]
scaler = MinMaxScaler()
df['total_of_directions'] = scaler.fit_transform(df[['total_of_directions']])
pedcountdata = []
for timestamp, group in df.groupby('timestamp'):
    pedcountdata.append(group[['lat', 'lon', 'total_of_directions']].values.tolist())
#pedcountdata

In [8]:
import logging
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Image
import populationVisual as pop
import congestion_statistic as tc
import path1 as pt
import ped_Visual as pc
import mastodon_helper as mtd
import importlib
import re
import folium
import webbrowser
from datetime import timedelta

importlib.reload(mtd)

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
# Create a title widget
title = widgets.HTML(value="<h3>Control Buttons for Visualization</h3>")
# Dataframe in Mastodon scenario
df1 = mtd.fetch_data("http://127.0.0.1:9090/mastodonquery", "df1")
df2 = mtd.fetch_data("http://127.0.0.1:9090/aussocialquery", "df2")
# Define the functions to generate the charts
def generate_population_chart(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            pop.bar_chart()  # This function generates and shows the plot
            logger.info("Population chart generated successfully")
    except Exception as e:
        logger.error(f"Error generating population chart: {e}")
        print(f"Error generating population chart: {e}")
# Creating widgets for interactive input
date_text = widgets.Text(
    description='Date',
    value='2024-05-28',
    placeholder='Enter date (YYYY-MM-DD)'
)

area_id_text = widgets.Text(
    description='Area ID',
    value='2644',
    placeholder='Enter area ID'
)
error_label = widgets.Label()
def generate_traffic_congestion_chart(d):
    try:
        with output:
            clear_output(wait=True)
            #clear_output(wait=True)  # Clear previous output
            date = date_text.value
            area_id = area_id_text.value
            if not re.match(r'^\d{4}-\d{2}-\d{2}$', date):
                error_label.value = "Error: Date must be in YYYY-MM-DD format"
            elif not area_id.isdigit():
                error_label.value = "Error: Area ID must be a number" 
            else:
                error_label.value = ""
                tc.plot_traffic_congestion(date, area_id)
            logger.info("Traffic congestion chart generated successfully")
    except Exception as e:
        logger.error(f"Error generating traffic congestion chart: {e}")
        print(f"Error generating traffic congestion chart: {e}")

def generate_mastodon_sentiment_activeness(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            mtd.plot_count_and_sentiment(df1)
            mtd.plot_count_and_sentiment(df2)
            logger.info("mastodon sentiment vs activeness chart generated successfully")
    except Exception as e:
        logger.error(f"Error generating mastodon sentiment vs activeness chart: {e}")
        print(f"Error generating mastodon sentiment vs activeness chart: {e}")

def generate_mastodon_pie_chart(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            mtd.plot_cumulative_pie_charts(df1)
            mtd.plot_cumulative_pie_charts(df2)
            logger.info("Mastodon Pie chart generated successfully")
    except Exception as e:
        logger.error(f"Error generating Mastodon Pie chart: {e}")
        print(f"Error generating Mastodon Pie chart: {e}")
def generate_mastodon_bar_chart(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            mtd.plot_comparision_bar(df1, df2)
            logger.info("Mastodon Bar chart generated successfully")
    except Exception as e:
        logger.error(f"Error generating Mastodon Bar chart: {e}")
        print(f"Error generating Mastodon Bar chart: {e}")

def generate_mastodon_correlation_matrix(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            mtd.plot_correlation_matrix(df1, 'mastodon.au')
            mtd.plot_correlation_matrix(df2, 'aus.social')
            logger.info("Mastodon Correlation Matrix generated successfully")
    except Exception as e:
        logger.error(f"Error generating Mastodon Correlation Matrix: {e}")
        print(f"Error generating Mastodon Correlation Matrix: {e}")
df11 = mtd.dfInternal(df1)
df22 = mtd.dfInternal(df2)
def generate_mastodon_residual_plot(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            mtd.evaluate_linear_regression(df11)
            mtd.evaluate_linear_regression(df22)
            logger.info("Mastodon residual plots generated successfully")
    except Exception as e:
        logger.error(f"Error generating Mastodon residual plots: {e}")
        print(f"Error generating Mastodon residual plots: {e}")

def generate_mastodon_scatterplot(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            mtd.plot_3d_scatter_with_regression(df11)
            mtd.plot_3d_scatter_with_regression(df22)
            logger.info("Mastodon Scatterplots generated successfully")
    except Exception as e:
        logger.error(f"Error generating Mastodon Scatterplots: {e}")
        print(f"Error generating Mastodon Scatterplots: {e}")
def generate_padcount_byDay(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            pc.plot_hourly_pedestrian_count(df_pedcountperh)
            logger.info("Pedestrians count by hour generated successfully")
    except Exception as e:
        logger.error(f"Error generating Pedestrians count by hour: {e}")
        print(f"Error generating Pedestrians count by hour: {e}")
def generate_padcount_bySensor(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            pc.plot_mean_pedestrian_by_sensor(df_pedcountperh)
            logger.info("Pedestrians count by sensor generated successfully")
    except Exception as e:
        logger.error(f"Error generating Pedestrians count by sensor: {e}")
        print(f"Error generating Pedestrians count by sensor: {e}")
def generate_lineServiceCount(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            display(Image(filename="trainservice/lineservicecounts.png"))
            logger.info("Service line bar chart generated successfully")
    except Exception as e:
        logger.error(f"Error generating Service line bar chart: {e}")
        print(f"Error generating Service line bar chart: {e}")
def generate_trainServiceHis(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            display(Image(filename="trainservice/trainservicehis.png"))
            logger.info("Train service histogram generated successfully")
    except Exception as e:
        logger.error(f"Error generating Train service histogram: {e}")
        print(f"Error generating Train service histogram: {e}")
def generate_suburbpopulation(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            display(Image(filename="trainservice/suburbpopulation.png"))
            logger.info("Suburb population generated successfully")
    except Exception as e:
        logger.error(f"Error generating Suburb population: {e}")
        print(f"Error generating Suburb population: {e}")
def generate_stationcountrank(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            display(Image(filename="trainservice/stationcountrank.jpeg"))
            logger.info("Station counts bar chart generated successfully")
    except Exception as e:
        logger.error(f"Error generating Station counts bar chart: {e}")
        print(f"Error generating Station counts bar chart: {e}")
def generate_linestopscounts(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            display(Image(filename="trainservice/linestopscounts.png"))
            logger.info("Line stops counts bar chart generated successfully")
    except Exception as e:
        logger.error(f"Error generating Line stops counts bar chart: {e}")
        print(f"Error generating Line stops counts bar chart: {e}")
def generate_congestion_map(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            m = folium.Map(location=[-37.8128,144.9675], zoom_start=10)
            api_url = "http://127.0.0.1:9090/realtimecongestion"
            output_file = "path_map_white_background2.html"
            pt.CongestionMap(api_url, output_file,m).generate_map()
            display(m)
            m.save("congestion map.html")
            logger.info("Traffic congestion map generated successfully")
    except Exception as e:
        logger.error(f"Error generating traffic congestion map: {e}")
        print(f"Error generating traffic congestion map: {e}")
def generate_population_map(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            m = folium.Map(location=[-37.8128,144.9675], zoom_start=10)
            pop.plot_population_map1(m)
            display(m)
            m.save("population map.html")
            logger.info("Regional population map generated successfully")
    except Exception as e:
        logger.error(f"Error generating Regional population map: {e}")
        print(f"Error generating Regional population map: {e}")
def generate_padcount_Sensor_map(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            m = folium.Map(location=[-37.8128,144.9675], zoom_start=10)
            file_path1="pedestrian-counting-system-sensor-locations.json"
            pc.create_sensor_map(file_path1,m)
            display(m)
            m.save("sensor map.html")
            logger.info("Pedestrians sensor map generated successfully")
    except Exception as e:
        logger.error(f"Error generating Pedestrians sensor map: {e}")
        print(f"Error generating Pedestrians sensor map: {e}")
def generate_padcount_map(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            m = folium.Map(location=[-37.8128,144.9675], zoom_start=10)
            pc.create_pedcount_map(pedcountdata,df_pedcountperh,m)
            display(m)
            m.save("pedcount map.html")
            logger.info("Pedestrians count map generated successfully")
    except Exception as e:
        logger.error(f"Error generating Pedestrians count map: {e}")
        print(f"Error generating Pedestrians count map: {e}")
def generate_line_railway_map(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            file_path = "C:/Users/jocel/Desktop/master/comp90024/a2/line_railway.html"
            webbrowser.open(f"file:///{file_path}")
            logger.info("Line railway map generated successfully")
    except Exception as e:
        logger.error(f"Error generating Line railway map: {e}")
        print(f"Error generating Line railway map: {e}")
def generate_all_stations_map(d):
    try:
        with output:
            clear_output(wait=True)  # Clear previous output
            file_path = "C:/Users/jocel/Desktop/master/comp90024/a2/all_station_info.html"
            webbrowser.open(f"file:///{file_path}")
            logger.info("All stations info map generated successfully")
    except Exception as e:
        logger.error(f"Error generating All stations info map: {e}")
        print(f"Error generating All stations info map: {e}")

# option box to choose maps to plot
option1 = widgets.Checkbox(value=False, description='Show Regional Population Map')
option2 = widgets.Checkbox(value=False, description='Show Sensor Station Map')
option3 = widgets.Checkbox(value=False, description='Show Traffic Congestion Map')

def generate_combined_map(d):
    try:
        with output:
            clear_output(wait=True)
            m = folium.Map(location=[-37.8128,144.9675], zoom_start=10)
            if option1.value:
                pop.plot_population_map1(m)
            if option2.value:
                file_path1="pedestrian-counting-system-sensor-locations.json"
                pc.create_sensor_map(file_path1,m)
            if option3.value:
                api_url = "http://127.0.0.1:9090/realtimecongestion"
                output_file = "path_map_white_background2.html"
                pt.CongestionMap(api_url, output_file,m).generate_map()
            # Save plot locally
            m.save("Combined maps.html")
            display(m)
            logger.info("Combined maps generated successfully")
    except Exception as e:
        logger.error(f"Error generating Combined maps {e}")
        print(f"Error generating Combined maps: {e}")
    


output = widgets.Output()
# Create buttons
population_button = widgets.Button(description="Generate Population Chart",layout=widgets.Layout(width='70%'))
population_map_button = widgets.Button(description="Generate Regional Population Map Chart",layout=widgets.Layout(width='70%'))
congestion_button = widgets.Button(description="Generate Traffic Congestion Chart with Date and ID:",layout=widgets.Layout(width='70%'))
congestion_map_button = widgets.Button(description="Generate Traffic Congestion Map",layout=widgets.Layout(width='70%'))
mastodon_sentiment_button = widgets.Button(description="Generate Activeness (doc_count) vs Sentiment Score (average_sentiment)",layout=widgets.Layout(width='70%'))
mastodon_pie_button = widgets.Button(description="Generate Pie Chart for Each Category",layout=widgets.Layout(width='70%'))
mastodon_bar_button = widgets.Button(description="Generate Bar Chart for Each Category",layout=widgets.Layout(width='70%'))
mastodon_correlation_button = widgets.Button(description="Generate Correlation Matrix for Mastodon",layout=widgets.Layout(width='70%'))
mastodon_residual_button = widgets.Button(description="Generate Residual Plot for Mastodon",layout=widgets.Layout(width='70%'))
mastodon_scatterplot_button = widgets.Button(description="Generate Scatterplot for Mastodon",layout=widgets.Layout(width='70%'))
pedcount_byDay_button = widgets.Button(description="Generate Bar chart for Pedestrian Counting by hour of the day",layout=widgets.Layout(width='70%'))
pedcount_bySensor_button = widgets.Button(description="Generate Bar chart for Pedestrian Counting by Sensor",layout=widgets.Layout(width='70%'))
pedcount_sensor_map_button = widgets.Button(description="Generate map for Sensor Station",layout=widgets.Layout(width='70%'))
pedcount_map_button = widgets.Button(description="Generate map for Pedestrian Distribution with time",layout=widgets.Layout(width='70%'))
service_lineCount_button = widgets.Button(description="Generate Line Service Count in bar chart",layout=widgets.Layout(width='70%'))
service_trainServiceHis_button = widgets.Button(description="Generate train service in histogram",layout=widgets.Layout(width='70%'))
service_population_button = widgets.Button(description="Generate Suburb Population in bar chart",layout=widgets.Layout(width='70%'))
service_stationcountrank_button = widgets.Button(description="Generate Station Count in bar chart",layout=widgets.Layout(width='70%'))
service_linestopscounts_button = widgets.Button(description="Generate Number of Stations by line in bar chart",layout=widgets.Layout(width='70%'))
service_line_railway_map_button = widgets.Button(description="Generate Map of railway line",layout=widgets.Layout(width='70%'))
service_all_stations_map_button = widgets.Button(description="Generate Map of all stations information",layout=widgets.Layout(width='70%'))
map_combine_button = widgets.Button(description='Draw combined maps in the choices below:',layout=widgets.Layout(width='70%'))

# Attach functions to buttons
population_button.on_click(generate_population_chart)
population_map_button.on_click(generate_population_map)
congestion_button.on_click(generate_traffic_congestion_chart)
congestion_map_button.on_click(generate_congestion_map)
mastodon_sentiment_button.on_click(generate_mastodon_sentiment_activeness)
mastodon_pie_button.on_click(generate_mastodon_pie_chart)
mastodon_bar_button.on_click(generate_mastodon_bar_chart)
mastodon_correlation_button.on_click(generate_mastodon_correlation_matrix)
mastodon_residual_button.on_click(generate_mastodon_residual_plot)
mastodon_scatterplot_button.on_click(generate_mastodon_scatterplot)
pedcount_byDay_button.on_click(generate_padcount_byDay)
pedcount_bySensor_button.on_click(generate_padcount_bySensor)
pedcount_sensor_map_button.on_click(generate_padcount_Sensor_map)
pedcount_map_button.on_click(generate_padcount_map)
service_lineCount_button.on_click(generate_lineServiceCount)
service_trainServiceHis_button.on_click(generate_trainServiceHis)
service_population_button.on_click(generate_suburbpopulation)
service_stationcountrank_button.on_click(generate_stationcountrank)
service_linestopscounts_button.on_click(generate_linestopscounts)
service_line_railway_map_button.on_click(generate_stationcountrank)
service_all_stations_map_button.on_click(generate_linestopscounts)
map_combine_button.on_click(generate_combined_map)

# Organize the layout
input_widgets = widgets.VBox([date_text, area_id_text, error_label])
congestion_section = widgets.VBox([congestion_map_button,congestion_button, input_widgets])
map_choice = widgets.VBox([map_combine_button,option1, option2, option3])
button_section = widgets.VBox([title,population_button, population_map_button, congestion_section, mastodon_sentiment_button,mastodon_pie_button,mastodon_bar_button,
                               mastodon_correlation_button,mastodon_residual_button,mastodon_scatterplot_button,pedcount_byDay_button,
                               pedcount_bySensor_button,pedcount_sensor_map_button,pedcount_map_button,service_lineCount_button,service_trainServiceHis_button,service_population_button,
                               service_stationcountrank_button,service_linestopscounts_button,service_line_railway_map_button,service_all_stations_map_button,map_choice])


# Display the organized layout
display(button_section,output)

df1 load succeeded.
df2 load succeeded.


Output()